# This notebook was used to measure the sensibility of the model

Required libraries: Ransom, Plots, Base.Threads, LateXStrings, Statistics, Plot.Measures and JLD2

Run the notebook top to bottom

**Warning:** Very computer heavy, recomended running on *Nuredunna* cluster with 6 cores

# Libraries, Functions & Parameters

Libraries

In [ ]:
using Random
using Plots
using Base.Threads
using LaTeXStrings
using Statistics
using Plots.Measures
using JLD2

Functions

In [ ]:
function laplacian(n)
  N = size(n, 1)
  lap = similar(n)

  for j in 1:N, i in 1:N
    ip = mod1(i+1, N)
    im = mod1(i-1, N)
    jp = mod1(j+1, N)
    jm = mod1(j-1, N)
    lap[i,j] = n[ip,j] + n[im,j] + n[i,jp] + n[i,jm] - 4*n[i,j]
  end

  return lap
end

function step(n,dt,omega,a,b,eps)
  lap = laplacian(n)
  dndt = omega.*n + a.*n.^2 - b.*n.^3 + eps*lap

  return n + dt*dndt
end

# Crab's effect on algae
function grazing(positions, N, α, σ)
  F = zeros(N, N)
  limit = σ*3 # 99.7% of the gaussian distribution

  for (cx, cy) in positions
    for i in cx-limit:cx+limit, j in cy-limit:cy+limit
      ii = mod1(i, N)
      jj = mod1(j, N)
      r2 = (i - cx)^2 + (j - cy)^2
      F[ii, jj] += exp(-r2 / (2*σ^2))
    end
  end

  return (α / (2π * σ^2)) * F
end

# Algae step function
function stepcrabs(n, dt, omega, a, b, eps, crab_positions, α, σ, ρa)
  lap   = laplacian(n)
  graze = grazing(crab_positions, size(n,1), α, σ)
  dndt = omega.*n + a.*n.^2 - b.*n.^3 + eps*lap - graze./ρa

  return max.(0.0, n + dt*dndt)
end

function biomass_available(cx, cy, n, N, ΔL, σ)
  G = 0.0

  for i in (cx - σ):(cx + σ)
    for j in (cy - σ):(cy + σ)
      # Physical distance
      dx = (i - cx) * ΔL
      dy = (j - cy) * ΔL
      r2 = dx^2 + dy^2

      if r2 <= σ^2
        ii = mod1(i, N)
        jj = mod1(j, N)
        G += exp(-r2 / (2 * σ^2)) * n[ii, jj]
      end
    end
  end

  return (ΔL^2 / (2π * σ^2)) * G
end

# Local resource abundance added
function crab_update_resource!(crab_positions, n, dt, L, ΔL, σ, ρc)
  nc_t = length(crab_positions)

  # Edge case: If there are no crabs, there is no breeding.
  if nc_t == 0
    return crab_positions
  end

  N = size(n, 1)
  ζ = max(0, 1-nc_t/(ρc*L^2)) # carrying capacity
  μ_max = 3 # average lifespan (years)
  new_crabs_count = 0
  alive = trues(nc_t)

  # Breeding and local mortality per crab
  for k in 1:nc_t
    (cx, cy) = crab_positions[k]

    G = biomass_available(cx, cy, n, N, ΔL, σ)
    P_breed = G * ζ # Breeding probability

    if rand() < P_breed
      mean_offspring = 10 * (dt / 365)

      # Stochastic rounding to get a discrete integer of offspring
      base_offspring = floor(Int, mean_offspring)
      prob_fraction = mean_offspring - base_offspring
      offspring = base_offspring + (rand() < prob_fraction ? 1 : 0)

      new_crabs_count += offspring
    end

    # Mortality
    μ_local = μ_max * max(1e-5, 1 - exp(-G))
    P_death = dt / (μ_local * 365)

    if rand() < P_death
      alive[k] = false
    end
  end

  crab_positions = crab_positions[alive]

  # Larval Settlement
  if new_crabs_count > 0
    # Find aviable spaces
    occupied = falses(N, N)

    for (cx, cy) in crab_positions
      for x in -6:6, y in -6:6
        if x^2 + y^2 <= 32 # Area of 1 m²
          px = mod1(cx + x, N)
          py = mod1(cy + y, N)
          occupied[px, py] = true
        end
      end
    end

    # Gather all empty coordinates
    empty_sites = Tuple{Int, Int}[]
    for i in 1:N, j in 1:N
      if !occupied[i, j]
        push!(empty_sites, (i, j))
      end
    end

    # Shuffle empty sites and pick the required amount (up to what's available)
    shuffle!(empty_sites)
    spawn_count = min(new_crabs_count, length(empty_sites))

    # Append the new juvenile crabs to the population
    for k in 1:spawn_count
      push!(crab_positions, empty_sites[k])
    end
  end

  return crab_positions
end

step (generic function with 1 method)

Project parameters

In [ ]:
# Lattice
L = 500 #dm
ΔL = 1 #dm
N = Int(L/ΔL) #lattice units
dt = 0.1 # day

# Wet to Dry convertion
WW_to_DW = 0.1

# Crabs
σ = 10 #dm
α = 3.83*WW_to_DW #[gr DW crab⁻¹ day⁻¹]
μ = 3 #years
ρc = 0.01 #[crabs dm⁻²]

# Algae (Computed in "Algae parameter ajustment")
ρa = 1 #[gr DW dm⁻²]
scale = 0.055
omega = -0.4 * scale
a = 1.4 * scale
b = 1.0 * scale
eps = 0.07

0.23

In [ ]:
Random.seed!(1)

TaskLocalRNG()

# NPP vs ϵ

### Main loop

In [ ]:
# Loop parameters [to be modified for combinations that take longer to reach stability]
scale_sweep = [0.033, 0.044, 0.055, 0.066, 0.077]
eps_sweep = [0.007, 0.014, 0.07, 0.35, 0.7]
repetitions = 10
max_years = 30

simulation_time = Int(3650*max_years)
years = (collect(0:2*3652.5:max_years*3652.5), string.(0:2:max_years))

# Results initialization
av_final_crabs = zeros(length(scale_sweep), length(eps_sweep))
std_final_crabs = copy(av_final_crabs)
av_final_biomass = copy(av_final_crabs)
std_final_biomass = copy(av_final_crabs)

# Thread-safe plotting
plot_lock = ReentrantLock()

@threads for s in eachindex(scale_sweep)
  scale = scale_sweep[s]
  omega = -0.4 * scale
  a = 1.4 * scale
  b = 1.0 * scale

  for (ϵ, eps) in enumerate(eps_sweep)
    biomass_density_m2 = zeros(repetitions, simulation_time)
    total_crabs = zeros(repetitions, simulation_time)
    final_crabs = zeros(repetitions)
    final_biomass = zeros(repetitions)

    for rep in 1:repetitions
      # Initial grid
      n = ones(N,N) * 0.7 + rand(N,N) * 0.2
      n_crabs = 20
      crab_positions = [(rand(1:N), rand(1:N)) for i in 1:n_crabs]
      first = true

      # Simulation
      for i in 1:simulation_time
        crab_positions = crab_update_resource!(crab_positions, n, dt, L, ΔL, σ, ρc)
        n = stepcrabs(n, dt, omega, a, b, eps, crab_positions, α, σ, ρa)

        biomass_density_m2[rep, i] = (sum(n) / L^2) * 100 # gr DW m⁻²
        total_crabs[rep, i] = length(crab_positions)
      end

      final_crabs[rep] = length(crab_positions) / (L/10)^2 * 100 # 100m⁻²
      final_biomass[rep] = (sum(n) / L^2) * 100 # gr DW m⁻²
    end

    # Snapshot of the trajectory
    av_biomass_density_m2 = vec(mean(biomass_density_m2, dims=1))
    std_biomass_density_m2 = vec(std(biomass_density_m2, dims=1))
    crab_density = total_crabs ./ (L/10)^2 * 100 # 100m⁻²
    av_crabs = vec(mean(crab_density, dims=1))
    std_crabs = vec(std(crab_density, dims=1))

    lock(plot_lock) do
      po = plot(
        1:simulation_time, av_biomass_density_m2,
        ribbon = std_biomass_density_m2,
        xlabel = L"Years",
        ylabel = L"Biomass\ density\ (gr\ DW\ m^{-2})",
        title = "NPP = $(round(scale * 5.45, digits = 2)) ϵ = $eps",
        ylims = (0, 100),
        color = :green,
        label = L"Biomass",
        legend = :topright,
        xticks = years
      )
      plot!(po, [NaN], [NaN], color = "#921b13", label = L"Percnon gibbesi")
      po_twin = twinx(po)
      plot!(po_twin,
        1:simulation_time, av_crabs,
        ribbon = std_crabs,
        ylabel = L"Crab\ density\ (100m^{-2})",
        ylims = (0, 100),
        color = "#921b13",
        label = false
      )
      savefig(po, "NPP vs ϵ - scale $scale ϵ $eps.png")
    end

    # Getting the results
    av_final_crabs[s, ϵ] = mean(final_crabs)
    std_final_crabs[s, ϵ] = std(final_crabs)
    av_final_biomass[s, ϵ] = mean(final_biomass)
    std_final_biomass[s, ϵ] = std(final_biomass)
  end
end

@save "Simulation_Sweep_Raw.jld2" av_final_crabs std_final_crabs av_final_biomass std_final_biomass

### Graphs

In [ ]:
NPP_sweep = [0.12, 0.18, 0.23, 0.28, 0.32]

ex = heatmap(string.(eps_sweep), string.(NPP_sweep), av_final_crabs',
  color = cgrad([:white, "#921b13"]),
  xlabel = L"Diffusion\ Rate\ (\epsilon)",
  ylabel = L"NPP\ (g\ DW\ m^{-2}\ day^{-1})",
  framestyle = :box,
  colorbar_title = "\n" * L"Percnon\ gibbesi\ density\ (m^{-2})",
  guidefontsize = 14,
  tickfontsize  = 10,
  colorbar_titlefontsize = 10,
  colorbar_tickfontsize  = 10,
  right_margin = 15mm,
  size=(480,360)
)

savefig(ex, "NPP vs ϵ - crabs.png")

ab = heatmap(string.(eps_sweep), string.(NPP_sweep), av_final_biomass',
  color = cgrad([:white, :green]),
  xlabel = L"Diffusion\ Rate\ (\epsilon)",
  ylabel = L"NPP\ (g\ DW\ m^{-2}\ day^{-1})",
  framestyle = :box,
  colorbar_title = "\n" * L"Biomass\ density\ (g\ DW\ m^{-2})",
  guidefontsize = 14,
  tickfontsize  = 10,
  colorbar_titlefontsize = 10,
  colorbar_tickfontsize  = 10,
  right_margin = 15mm,
  size=(480,360)
)

savefig(ab, "NPP vs ϵ - biomass.png")

sb = heatmap(string.(eps_sweep), string.(NPP_sweep), std_final_biomass',
  color = cgrad([:white, :green]),
  xlabel = L"Diffusion\ Rate\ (\epsilon)",
  ylabel = L"NPP\ (g\ DW\ m^{-2}\ day^{-1})",
  framestyle = :box,
  colorbar_title = "\n" * L"Biomass\ standard\ deviation",
  guidefontsize = 14,
  tickfontsize  = 10,
  colorbar_titlefontsize = 10,
  colorbar_tickfontsize  = 10,
  right_margin = 15mm,
  size=(480,360)
)

savefig(sb, "NPP vs ϵ - biomass standard deviation.png")

sc = heatmap(string.(eps_sweep), string.(NPP_sweep), std_final_crabs',
  color = cgrad([:white, "#921b13"]),
  xlabel = L"Diffusion\ Rate\ (\epsilon)",
  ylabel = L"NPP\ (g\ DW\ m^{-2}\ day^{-1})",
  framestyle = :box,
  colorbar_title = "\n\n" * L"Percnon\ gibbesi\ standard\ deviation",
  guidefontsize = 14,
  tickfontsize  = 10,
  colorbar_titlefontsize = 10,
  colorbar_tickfontsize  = 10,
  right_margin = 15mm,
  size=(480,360)
)

savefig(sc, "NPP vs ϵ - percnon gibbesi standard deviation.png")

biomass_per_crab = [c > 0 ? (b / c) : NaN for (b, c) in zip(av_final_biomass, av_final_crabs)]

bc = heatmap(string.(eps_sweep), string.(NPP_sweep), biomass_per_crab',
  color = cgrad(["#921b13", :green]),
  xlabel = L"Diffusion\ Rate\ (\epsilon)",
  ylabel = L"NPP\ (g\ DW\ m^{-2}\ day^{-1})",
  framestyle = :box,
  colorbar_title = "\n" * L"Biomass\ to\ crab\ ratio\ (g\ DW\ m^{-2}\ crab^{-1})",
  guidefontsize = 14,
  tickfontsize  = 10,
  colorbar_titlefontsize = 10,
  colorbar_tickfontsize  = 10,
  right_margin = 15mm,
  size=(480,360)
)

savefig(bc, "NPP vs ϵ - biomass per crab.png")

GKS: Rectangle definition is invalid in routine SET_WINDOW
GKS: Rectangle definition is invalid in routine CELLARRAY
invalid range


"/content/NPP vs ϵ - biomass per crab.png"